# Marker selection

After running all the prompt versions we now select an ideal prompt of our 5 versions

## Selection 

We select the marker with the best mean F1 across all three providers. We note anything outstanding or weird to answer or error rate since they will be excluded from the F1 score. We care equally about both FN and FP cases, we therefore use the macro F1. The argument which of the two cases matter more to a geocoding verifier can be made separately, for consistency purposes we account for both here.

In [1]:
import hashlib
from pathlib import Path

import duckdb
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

from thesis.config import load_config
from thesis.vlm.vlm_adapter import VLMResponse

prompts = {p["id"]: p["text"] for p in load_config("prompts")["prompts"]}
roleplaying_prompt = prompts["roleplaying"]
role_sha = hashlib.sha256(roleplaying_prompt.encode()).hexdigest()
prompt_lookup = {role_sha: "roleplaying"}

ablation_dir = ablation_dir = Path("/Users/phillip/repositories/bachelor-thesis/data/ablation")

In [2]:
responses = []
for provider_dir in (
    Path("/Users/phillip/repositories/bachelor-thesis/data/vlm_cache") / p for p in ("openai", "anthropic", "google")
):  # noqa: E501
    for f in provider_dir.glob("*.json"):
        r = VLMResponse.model_validate_json(f.read_text())
        responses.append(
            {
                "provider": r.provider,
                "image_sha256": r.image_sha256,
                "prompt_id": prompt_lookup.get(r.prompt_sha256),
                "parsed_answer": r.parsed_answer,
                "refusal_error_state": r.refusal_error_state,
            }
        )
responses_df = pd.DataFrame(responses)
responses_df = responses_df[responses_df["prompt_id"] == "roleplaying"]

con = duckdb.connect(str(ablation_dir / "manifest.duckdb"), read_only=True)
input_samples = con.sql(
    "SELECT * FROM samples",
).df()
input_samples["image_sha256"] = input_samples["image_path"].apply(
    lambda p: hashlib.sha256((ablation_dir / str(p)).read_bytes()).hexdigest()
)


input_samples["image_sha256"].duplicated().sum()  # 45?

np.int64(45)

In [ ]:
# dupes = input_samples[input_samples["image_sha256"].duplicated(keep=False)].sort_values("image_sha256")
# dupes[["sample_id", "image_sha256", "target_signed_edge_distance_m", "ground_truth_inside"]]
# same as w prompts

We found 5 hash collision, meaning images were the same. It shows that this is the case for -5m offset images vs control images. This means that -5m was the same as not shifting the centered pin. No mismatch or labeling or quality issue. Fine 

In [3]:
clean_samples = input_samples.sort_values("include_in_distance_curve", ascending=False).drop_duplicates(
    "image_sha256", keep="first"
)

merged = pd.merge(responses_df, clean_samples, on="image_sha256", validate="many_to_one")

In [4]:
merged["answered"] = merged["parsed_answer"].isin(["yes", "no"])
merged["predicted_positive"] = merged["parsed_answer"] == "yes"

rows = []
for (provider, shape, color), group in merged.groupby(["provider", "marker_shape", "marker_color"]):
    answered = group[group["answered"]]
    precision, recall, f1, _ = precision_recall_fscore_support(
        answered["ground_truth_inside"], answered["predicted_positive"], average="macro", zero_division=0
    )
    rows.append(
        {
            "provider": provider,
            "marker_shape": shape,
            "marker_color": color,
            "response_rate": group["answered"].mean(),
            "f1_macro": f1,
            "n": len(group),
        }
    )
marker_summary = pd.DataFrame(rows)
avg_macro_f1 = marker_summary.groupby(["marker_shape", "marker_color"])["f1_macro"].mean()

In [5]:
marker_summary

,provider,marker_shape,marker_color,response_rate,f1_macro,n
0,anthropic,crosshair,"(0, 130, 255)",0.907646,0.596075,1007
1,anthropic,crosshair,"(230, 20, 20)",0.934459,0.596694,1007
2,anthropic,crosshair,"(30, 255, 130)",0.929494,0.619428,1007
3,anthropic,dot,"(0, 130, 255)",0.922542,0.558324,1007
4,anthropic,dot,"(230, 20, 20)",0.913605,0.600265,1007
5,anthropic,dot,"(30, 255, 130)",0.921549,0.553894,1007
6,anthropic,google_pin,"(0, 130, 255)",0.925521,0.588824,1007
7,anthropic,google_pin,"(230, 20, 20)",0.932473,0.616208,1007
8,anthropic,google_pin,"(30, 255, 130)",0.930487,0.591201,1007
9,google,crosshair,"(0, 130, 255)",1.000000,0.816333,1007


In [ ]:
avg_macro_f1 = marker_summary.groupby(["marker_shape", "marker_color"])["f1_macro"].mean()
refusal_rates_provider = marker_summary.groupby("provider")["response_rate"].mean()
refusal_rates_prompt = marker_summary.groupby(["marker_shape", "marker_color"])["response_rate"].mean()

print("Average macro F1 by marker color and shape:")
print(avg_macro_f1)
print("\nAverage refusal rate by provider:")
print(refusal_rates_provider)
print("\nAverage refusal rate by prompt:")
print(refusal_rates_prompt)

Average macro F1 by provider:
marker_shape  marker_color  
crosshair     (0, 130, 255)     0.693702
              (230, 20, 20)     0.702050
              (30, 255, 130)    0.703212
dot           (0, 130, 255)     0.680112
              (230, 20, 20)     0.682900
              (30, 255, 130)    0.678352
google_pin    (0, 130, 255)     0.682758
              (230, 20, 20)     0.708979
              (30, 255, 130)    0.693515
Name: f1_macro, dtype: float64

Average refusal rate by provider:
provider
anthropic    0.924197
google       1.000000
openai       1.000000
Name: response_rate, dtype: float64

Average refusal rate by prompt:
marker_shape  marker_color  
crosshair     (0, 130, 255)     0.969215
              (230, 20, 20)     0.978153
              (30, 255, 130)    0.976498
dot           (0, 130, 255)     0.974181
              (230, 20, 20)     0.971202
              (30, 255, 130)    0.973850
google_pin    (0, 130, 255)     0.975174
              (230, 20, 20)     0.977491
     

Under the chosen criterion the ideal pin is `(google_pin, [230, 20, 20])`. It needs to be noted that the chosen marker had no effects on response or error rates. However the margins are thin. the google pin only marginally won against crosshair. The shape also matter more than color, the dot is a bad choice. 

Further qualitative analysis needed. This is just to lock the criterion.